# Metacatalog reliability tiers

Build **cleaned** (soft exclude) and **gold** (hard include) subsets from an existing
metacatalog + LST/sources tree under `CATALOG_DIR`.

**Semantics (grill-me locked):**
- Soft exclude removes only on *positive finite* failures; missing evidence keeps the row.
- Hard include requires every gate to be *calculable* (non-NaN) and pass.
- Multi-image: `n_lst ≥ 2` **or** ≥2 bands with unique association (`n_assoc==1`; Full counts).
- Residuals: absolute Jy/beam on the **origin-band seeded LST row** only (dual RMS/mean thresh, default 1.0).
- Jitter: seed-band rematch members; fail if RMS `> 0.3 × BMAJ`.
- α corner cuts are **not** used for selection.
- Both products are first-class; contract `gold ⊆ cleaned`.

Requires `lwa-catalog[analyze]` (`healpy` for binning, `lwa-healpix` for HiPS export).
The **HiPS sky viewer** at the end also needs `lwa-catalog[viz]` (`panel` + `ipyaladin`).

Set ``REUSE_CACHED_RELIABILITY = True`` (default) to load existing
``metacatalog_cleaned.parquet`` / ``metacatalog_gold.parquet`` (and existing HiPS
directories) instead of re-filtering or writing any output files.

**Run cells in order.**


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from lwa_catalog.analyze import (
    ReliabilityConfig,
    ReliabilityResult,
    filter_metacatalog_reliability,
    metacatalog_to_healpix,
    write_healpix_hips,
)
from lwa_catalog.io import read_all_lst_merged, read_metacatalog, read_table, write_table
from lwa_catalog.paths import CatalogLayout

# --- operator config ---
CATALOG_DIR = Path("/fast/claw/metacatalog_coaddR-0.75")  # existing fusion tree
NSIDE = 512
REUSE_CACHED_RELIABILITY = False  # load cached Parquet/HiPS; skip all writes when True
# HiPS output directory name under CATALOG_DIR; fields: {name} (full|cleaned|gold), {nside}
HIPS_DIR_TEMPLATE = "metacatalog_coaddR-0.75_{name}.hips"
CONFIG = ReliabilityConfig(
    resid_rms_thresh_jy=1.0,
    resid_mean_thresh_jy=1.0,
    jitter_bmaj_frac=0.3,
    strict=False,  # True → raise if gold ⊈ cleaned
)

layout = CatalogLayout(CATALOG_DIR)
PATHS = {
    "cleaned": layout.root / "metacatalog_cleaned.parquet",
    "gold": layout.root / "metacatalog_gold.parquet",
    "cleaned_flags": layout.root / "metacatalog_cleaned_flags.parquet",
    "gold_flags": layout.root / "metacatalog_gold_flags.parquet",
}
print("CATALOG_DIR =", layout.root.resolve())
print("REUSE_CACHED_RELIABILITY =", REUSE_CACHED_RELIABILITY)
print("HIPS_DIR_TEMPLATE =", HIPS_DIR_TEMPLATE)


CATALOG_DIR = /fast/claw/metacatalog_coaddR-0.75
REUSE_CACHED_RELIABILITY = False
HIPS_DIR_TEMPLATE = metacatalog_coaddR-0.75_{name}.hips


## Load metacatalog + LST-merged cache

In [2]:
metacatalog = read_metacatalog(layout)
print(f"metacatalog rows: {len(metacatalog)}")

_cache_ready = (
    REUSE_CACHED_RELIABILITY
    and PATHS["cleaned"].is_file()
    and PATHS["gold"].is_file()
)
if _cache_ready:
    lst_merged = None
    print("Cache hit for cleaned/gold — skipping LST-merged load")
else:
    lst_merged = read_all_lst_merged(layout)
    for band, df in lst_merged.items():
        print(f"  LST {band}: {len(df)} rows")


metacatalog rows: 106426
  LST Full: 52928 rows
  LST Blue: 105136 rows
  LST Green: 58546 rows
  LST Red: 23667 rows


## Load or run cleaned + gold filters

With ``REUSE_CACHED_RELIABILITY=True`` (default), load ``metacatalog_cleaned.parquet`` /
``metacatalog_gold.parquet`` (+ flags if present). If cache files are missing, filters
still run in memory but **no Parquet or HiPS files are written**. Set the flag to
``False`` to regenerate and write outputs.


In [3]:
def _reliability_from_parquet(catalog_path: Path, flags_path: Path) -> ReliabilityResult:
    cat = read_table(catalog_path)
    flags = read_table(flags_path) if flags_path.is_file() else pd.DataFrame()
    if "meta_id" in cat.columns:
        meta_ids = cat["meta_id"].to_numpy(dtype=int)
    else:
        meta_ids = np.arange(len(cat), dtype=int)
    return ReliabilityResult(
        catalog=cat,
        meta_ids=meta_ids,
        tier_counts=pd.DataFrame(columns=["tier", "n_in", "n_out", "n_removed"]),
        flags=flags,
        warnings=[f"loaded from {catalog_path.name}"],
    )


_cache_ready = (
    REUSE_CACHED_RELIABILITY
    and PATHS["cleaned"].is_file()
    and PATHS["gold"].is_file()
)
if _cache_ready:
    cleaned = _reliability_from_parquet(PATHS["cleaned"], PATHS["cleaned_flags"])
    gold = _reliability_from_parquet(PATHS["gold"], PATHS["gold_flags"])
    print(f"Loaded cached cleaned={len(cleaned.catalog)}  gold={len(gold.catalog)}")
else:
    if REUSE_CACHED_RELIABILITY:
        print("Cache missing — regenerating cleaned/gold…")
    cleaned, gold = filter_metacatalog_reliability(
        metacatalog,
        layout,
        config=CONFIG,
        lst_merged=lst_merged,
    )

print("=== cleaned (soft exclude) ===")
display(cleaned.tier_counts)
print(f"n_cleaned = {len(cleaned.catalog)}")

print("\n=== gold (hard include) ===")
display(gold.tier_counts)
print(f"n_gold = {len(gold.catalog)}")

assert set(gold.meta_ids).issubset(set(cleaned.meta_ids)), "gold ⊆ cleaned violated"
print("nesting OK: gold ⊆ cleaned")

display(cleaned.catalog.head(5))
display(gold.catalog.head(5))


=== cleaned (soft exclude) ===


,tier,n_in,n_out,n_removed
0,E0,106426,106426,0
1,E1,106426,90957,15469
2,E2,90957,90790,167
3,E3,90790,89965,825
4,E4,89965,83245,6720


n_cleaned = 83245

=== gold (hard include) ===


,tier,n_in,n_out,n_removed
0,I0,106426,106426,0
1,I1,106426,90957,15469
2,I2,90957,90790,167
3,I3,90790,89965,825
4,I4,89965,83245,6720
5,I5,83245,80824,2421


n_gold = 80824
nesting OK: gold ⊆ cleaned


,origin_band,bands_present,RA,DEC,Peak_flux,Total_flux,Maj,Min,PA,DC_Maj,...,Total_flux_Full,E_Total_flux_Full,RA_Full,DEC_Full,Maj_Full,Min_Full,PA_Full,DC_Maj_Full,DC_Min_Full,DC_PA_Full
26,Full,"Full,Green,Red",286.881839,7.139315,49.017454,73.086603,0.191557,0.180881,41.540364,0.112924,...,73.086603,2.549851,286.881839,7.139315,0.191557,0.180881,41.540364,0.112924,0.100684,22.501088
31,Full,"Full,Blue,Green,Red",287.779034,9.110626,44.784349,66.661460,0.187099,0.180474,47.747990,0.109159,...,66.661460,2.365978,287.779034,9.110626,0.187099,0.180474,47.747990,0.109159,0.100667,155.534356
59,Green,"Green,Red",152.735498,6.416151,36.949936,48.473720,0.175609,0.168468,37.511324,0.091983,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
68,Full,"Full,Blue,Green,Red",284.842552,12.986763,34.709492,53.501895,0.185169,0.182859,44.575855,0.113431,...,53.501895,2.087200,284.842552,12.986763,0.185169,0.182859,44.575855,0.113431,0.104082,135.029700
76,Full,"Full,Blue,Green,Red",285.702609,-23.499407,33.683914,52.005438,0.330845,0.232580,29.011296,0.220305,...,52.005438,3.112036,285.702609,-23.499407,0.330845,0.232580,29.011296,0.220305,0.110461,158.251987


,origin_band,bands_present,RA,DEC,Peak_flux,Total_flux,Maj,Min,PA,DC_Maj,...,Total_flux_Full,E_Total_flux_Full,RA_Full,DEC_Full,Maj_Full,Min_Full,PA_Full,DC_Maj_Full,DC_Min_Full,DC_PA_Full
26,Full,"Full,Green,Red",286.881839,7.139315,49.017454,73.086603,0.191557,0.180881,41.540364,0.112924,...,73.086603,2.549851,286.881839,7.139315,0.191557,0.180881,41.540364,0.112924,0.100684,22.501088
31,Full,"Full,Blue,Green,Red",287.779034,9.110626,44.784349,66.661460,0.187099,0.180474,47.747990,0.109159,...,66.661460,2.365978,287.779034,9.110626,0.187099,0.180474,47.747990,0.109159,0.100667,155.534356
59,Green,"Green,Red",152.735498,6.416151,36.949936,48.473720,0.175609,0.168468,37.511324,0.091983,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
68,Full,"Full,Blue,Green,Red",284.842552,12.986763,34.709492,53.501895,0.185169,0.182859,44.575855,0.113431,...,53.501895,2.087200,284.842552,12.986763,0.185169,0.182859,44.575855,0.113431,0.104082,135.029700
76,Full,"Full,Blue,Green,Red",285.702609,-23.499407,33.683914,52.005438,0.330845,0.232580,29.011296,0.220305,...,52.005438,3.112036,285.702609,-23.499407,0.330845,0.232580,29.011296,0.220305,0.110461,158.251987


## Write subsets + companion flags (never overwrite `metacatalog.parquet`)

In [4]:
out = layout.root
paths = PATHS
assert paths["cleaned"].name != "metacatalog.parquet"

if REUSE_CACHED_RELIABILITY:
    print("Skipping write — REUSE_CACHED_RELIABILITY=True")
    for label, p in paths.items():
        status = "ok" if p.is_file() else "missing"
        print(f"  {label}: {p} ({status})")
else:
    write_table(cleaned.catalog, paths["cleaned"])
    write_table(gold.catalog, paths["gold"])
    write_table(cleaned.flags, paths["cleaned_flags"])
    write_table(gold.flags, paths["gold_flags"])
    for label, p in paths.items():
        print(f"{label}: {p} ({p.stat().st_size} bytes)")


cleaned: /fast/claw/metacatalog_coaddR-0.75/metacatalog_cleaned.parquet (38033955 bytes)
gold: /fast/claw/metacatalog_coaddR-0.75/metacatalog_gold.parquet (36662235 bytes)
cleaned_flags: /fast/claw/metacatalog_coaddR-0.75/metacatalog_cleaned_flags.parquet (4502950 bytes)
gold_flags: /fast/claw/metacatalog_coaddR-0.75/metacatalog_gold_flags.parquet (4502950 bytes)


## Peak_flux Gaussian HEALPix → HiPS

Paints each source as an **elliptical Gaussian** on an equatorial HEALPix map
(`Peak_flux` amplitude; `Maj`/`Min` FWHM in degrees; `PA` from North toward East),
then writes a **HiPS** tile set with `lwa_healpix.healpix_to_hips` (not FITS).
Use `profile="point"` for single-pixel deposits.

Install: `pip install 'lwa-catalog[analyze]'` (`healpy` + `lwa-healpix`).

View with Aladin Lite: serve the output directory over HTTP and open `index.html`.
HiPS output directories must be empty (or new); remove old dirs before re-running.

Skipped when ``REUSE_CACHED_RELIABILITY=True`` (uses existing ``.hips`` directories).

In [5]:
# Peak_flux Gaussian HEALPix → HiPS (via lwa-healpix)
# Install: pip install 'lwa-catalog[analyze]'  (healpy + lwa-healpix)
# HiPS dirs must be empty; delete previous outputs before re-running.

out = layout.root
hips_dirs = {}
tier_cats = (
    ("full", metacatalog),
    ("cleaned", cleaned.catalog),
    ("gold", gold.catalog),
)

if REUSE_CACHED_RELIABILITY:
    print("Skipping HiPS export — REUSE_CACHED_RELIABILITY=True")
    for name, _cat in tier_cats:
        hips_dir = out / HIPS_DIR_TEMPLATE.format(name=name, nside=NSIDE)
        hips_dirs[name] = hips_dir
        status = "ok" if (hips_dir / "properties").is_file() else "missing"
        print(f"  {name}: {hips_dir} ({status})")
else:
    for name, cat in tier_cats:
        m = metacatalog_to_healpix(
            cat,
            nside=NSIDE,
            weight_col="Peak_flux",
            profile="gaussian",  # Maj/Min FWHM (deg), PA N→E; use "point" for single-pixel
        )
        hips_dir = out / HIPS_DIR_TEMPLATE.format(name=name, nside=NSIDE)
        hips_dirs[name] = write_healpix_hips(
            m,
            hips_dir,
            nest=False,
            coord_frame="equatorial",
            threads=True,
            properties={"obs_title": f"metacatalog {name} (Peak_flux Gaussians)"},
        )
        print(
            f"{name}: peak_max={float(m.max()):.4g} sum={float(m.sum()):.4g} "
            f"→ {hips_dirs[name]}"
        )

print(
    "\nServe a HiPS directory over HTTP and open index.html, e.g.:\n"
    f"  python -m http.server 8000 --directory {hips_dirs['gold']}\n"
    "  then browse http://localhost:8000/"
)


full: peak_max=234 sum=1.681e+06 → /fast/claw/metacatalog_coaddR-0.75/metacatalog_coaddR-0.75_full.hips
cleaned: peak_max=44.75 sum=7.904e+05 → /fast/claw/metacatalog_coaddR-0.75/metacatalog_coaddR-0.75_cleaned.hips
gold: peak_max=44.75 sum=7.451e+05 → /fast/claw/metacatalog_coaddR-0.75/metacatalog_coaddR-0.75_gold.hips

Serve a HiPS directory over HTTP and open index.html, e.g.:
  python -m http.server 8000 --directory /fast/claw/metacatalog_coaddR-0.75/metacatalog_coaddR-0.75_gold.hips
  then browse http://localhost:8000/


## HiPS sky viewer

Interactive **Aladin Lite** view with independent **Catalog** and **HiPS** dropdowns (mix and match).
Band-colored ellipse overlays are on by default. Requires `lwa-catalog[viz]` (`panel` + `ipyaladin`).

HiPS tiles load in your **web browser** (not the Jupyter kernel). Serve the catalog tree or
individual `.hips` directories over HTTP and set `HIPS_SERVER` to a URL the browser can reach.

Defaults: catalog = `metacatalog`, HiPS = `full`. Adjust **Max sources in view** if the overlay
is truncated in crowded fields. The overlay refreshes when you pan or zoom. For QA screenshots,
use `aladin.save_view_as_image(path)` on the underlying `ipyaladin` widget (see query notebook
**Save sky PNG** for an example).

In [7]:
import json
import threading
import urllib.error
import urllib.request

import astropy.units as u
import panel as pn
import param
from astropy.coordinates import SkyCoord

from lwa_catalog.constants import COLOR_BANDS
from lwa_catalog.io import read_lst_merged
from lwa_catalog.viz.aladin import overlay_catalog_by_band

# HiPS tiles load in your web browser (not the Jupyter kernel).
# HIPS_LIST_SERVER — survey names from ``cgi-bin/list-hips.py`` (dropdown; kernel fetch).
# HIPS_SERVER — tile base URL passed to Aladin (browser fetch; can differ from list host).
# If the list fetch fails, scans CATALOG_DIR for HiPS ``properties`` directories.
# Example: serve coadd tiles locally:
#   python -m http.server 8000 --directory /fast/claw/metacatalog_coadd2
#   HIPS_LIST_SERVER = "http://localhost:8000"
#   HIPS_SERVER = "http://localhost:8000"
HIPS_LIST_SERVER = "http://lwacalim10:3005"  # or http://localhost:3005 when SSH-tunneled
HIPS_SERVER = "http://localhost:3005"
DEFAULT_HIPS_SURVEY = HIPS_DIR_TEMPLATE.format(name="full", nside=NSIDE)
DEFAULT_CATALOG = "metacatalog"
SKY_FOV_DEG = 10.0
HIPS_VIEW_HEIGHT = 700
OVERLAY_MAX_SOURCES_DEFAULT = 1000
OVERLAY_MAX_SOURCES_BOUNDS = (100, 2000)

CATALOG_KEYS: list[str] = [
    "metacatalog",
    "cleaned",
    "gold",
    *(f"lst_{band}" for band in COLOR_BANDS),
]
CATALOG_LABELS: dict[str, str] = {
    "metacatalog": "metacatalog",
    "cleaned": "cleaned",
    "gold": "gold",
    **{f"lst_{band}": f"LST merged {band}" for band in COLOR_BANDS},
}

pn.extension(throttled=True)


def discover_local_hips_surveys(catalog_dir: Path = layout.root) -> list[str]:
    """Find HiPS tile directories under ``catalog_dir`` (dirs containing ``properties``)."""
    root = Path(catalog_dir)
    if not root.is_dir():
        return []
    return sorted(
        child.name
        for child in root.iterdir()
        if child.is_dir() and (child / "properties").is_file()
    )


def fetch_hips_surveys(
    list_base: str = HIPS_LIST_SERVER,
    *,
    catalog_dir: Path = layout.root,
    timeout: float = 5.0,
) -> list[str]:
    """Return HiPS survey names from ``list_base``, with local catalog fallback."""
    local = discover_local_hips_surveys(catalog_dir)
    url = f"{list_base.rstrip('/')}/cgi-bin/list-hips.py"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            remote = [str(s) for s in json.loads(resp.read().decode())]
    except (OSError, urllib.error.URLError, json.JSONDecodeError, TimeoutError) as exc:
        print(f"Warning: could not fetch HiPS surveys from {url}: {exc}")
        if local:
            print(
                f"Using {len(local)} HiPS director{'y' if len(local) == 1 else 'ies'} "
                f"under {catalog_dir}"
            )
            return local
        print(f"Falling back to DEFAULT_HIPS_SURVEY={DEFAULT_HIPS_SURVEY!r}")
        return [DEFAULT_HIPS_SURVEY]
    return sorted({*remote, *local})


def default_hips_survey(surveys: list[str]) -> str:
    """Pick a sensible default HiPS survey from the server list."""
    if DEFAULT_HIPS_SURVEY in surveys:
        return DEFAULT_HIPS_SURVEY
    for key in ("metacatalog_coadd2_full", "metacatalog_coadd2_gold", "metacatalog_coadd2_cleaned"):
        hit = next((s for s in surveys if key in s), None)
        if hit is not None:
            return hit
    return surveys[0] if surveys else DEFAULT_HIPS_SURVEY


def hips_survey_url(survey: str, *, base: str = HIPS_SERVER) -> str:
    """Build the HiPS root URL passed to ipyaladin (trailing slash for Aladin Lite)."""
    survey = survey.strip().strip("/")
    if survey.startswith("http://") or survey.startswith("https://"):
        return survey if survey.endswith("/") else f"{survey}/"
    return f"{base.rstrip('/')}/{survey}/"


def parse_coordinate(text: str) -> SkyCoord:
    """Parse a single sky position from free-form text."""
    text = text.strip()
    if not text:
        raise ValueError("Coordinate string is empty")
    parts = text.replace(",", " ").split()
    if len(parts) == 2:
        try:
            ra = float(parts[0])
            dec = float(parts[1])
            return SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")
        except ValueError:
            pass
    return SkyCoord(text, frame="icrs")


def catalog_overlay_name(catalog_key: str) -> str:
    """Parquet-style name passed to overlay band resolution."""
    if catalog_key.startswith("lst_"):
        return f"metacatalog_lst_{catalog_key[4:]}"
    return catalog_key


def load_catalog(catalog_key: str, cache: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Load a catalog DataFrame, with in-notebook caching for LST-merged bands."""
    if catalog_key in cache:
        return cache[catalog_key]
    if catalog_key == "metacatalog":
        df = metacatalog
    elif catalog_key == "cleaned":
        df = cleaned.catalog
    elif catalog_key == "gold":
        df = gold.catalog
    elif catalog_key.startswith("lst_"):
        df = read_lst_merged(layout, catalog_key[4:])
    else:
        msg = f"Unknown catalog key: {catalog_key!r}"
        raise KeyError(msg)
    cache[catalog_key] = df
    return df


class ReliabilityHiPSViewer(pn.viewable.Viewer):
    """Mix-and-match catalog + HiPS sky view with band-colored overlays."""

    catalog = param.Selector(default=DEFAULT_CATALOG, objects=CATALOG_KEYS)
    show_overlay = param.Boolean(default=True, doc="Draw catalog sources on the HiPS view")
    max_sources = param.Integer(
        default=OVERLAY_MAX_SOURCES_DEFAULT,
        bounds=OVERLAY_MAX_SOURCES_BOUNDS,
        doc="Max sources drawn inside the current FOV",
    )
    coordinate = param.String(
        default="83.633 -5.391",
        doc="Center coordinate (decimal deg or sexagesimal)",
    )

    def __init__(self, **params):
        super().__init__(**params)
        self._catalog_cache: dict[str, pd.DataFrame] = {}
        self._hips_current_survey = ""
        self._overlay_refresh_timer: threading.Timer | None = None
        self._ready = False

        hips_surveys = fetch_hips_surveys()
        hips_default = default_hips_survey(hips_surveys)
        init_coord = parse_coordinate(self.coordinate)

        from ipyaladin import Aladin

        self._catalog_w = pn.widgets.Select.from_param(self.param.catalog, name="Catalog")
        self._hips_survey_w = pn.widgets.Select(
            name="HiPS survey",
            options=hips_surveys,
            value=hips_default,
            sizing_mode="stretch_width",
        )
        self._hips_fov_w = pn.widgets.FloatSlider(
            name="FOV (deg)",
            start=1,
            end=90.0,
            value=SKY_FOV_DEG,
            step=0.05,
        )
        self._overlay_w = pn.widgets.Checkbox.from_param(
            self.param.show_overlay,
            name="Show catalog overlay",
        )
        self._max_sources_w = pn.widgets.IntInput.from_param(
            self.param.max_sources,
            name="Max sources in view",
            width=160,
        )
        self._coord_w = pn.widgets.TextInput.from_param(
            self.param.coordinate, name="Coordinate", placeholder="RA Dec"
        )
        self._center_btn = pn.widgets.Button(name="Center view", button_type="primary")
        self._center_btn.on_click(self._on_center_click)

        self._hips_status = pn.pane.Markdown("", sizing_mode="stretch_width")
        self._overlay_status = pn.pane.Markdown("", sizing_mode="stretch_width")

        self._hips_current_survey = hips_survey_url(hips_default)
        self._aladin = Aladin(
            survey=self._hips_current_survey,
            target=init_coord,
            fov=SKY_FOV_DEG,
            height=HIPS_VIEW_HEIGHT,
        )
        self._hips_view = pn.pane.IPyWidget(
            self._aladin,
            height=HIPS_VIEW_HEIGHT + 20,
            sizing_mode="stretch_width",
        )
        self._aladin.observe(self._on_aladin_view_trait, names=["_target", "_fov"])

        self._catalog_w.param.watch(self._on_catalog_change, "value")
        self._hips_survey_w.param.watch(self._on_hips_survey_change, "value")
        self._hips_fov_w.param.watch(self._on_hips_fov_change, "value")
        self._overlay_w.param.watch(self._on_overlay_change, "value")
        self._max_sources_w.param.watch(self._on_max_sources_change, "value")

        self._panel = pn.Column(
            pn.pane.Markdown("### Sky context (HiPS + catalog overlay)", disable_anchors=True),
            pn.Row(self._catalog_w, self._hips_survey_w),
            pn.Row(self._hips_fov_w, self._overlay_w, self._max_sources_w),
            pn.Row(self._coord_w, self._center_btn),
            self._hips_status,
            self._overlay_status,
            self._hips_view,
        )

        self._ready = True
        self._update_sky_view(init_coord)

    def __panel__(self):
        return self._panel

    def _view_coord(self) -> SkyCoord:
        return parse_coordinate(self.coordinate)

    def _overlay_view(self) -> tuple[SkyCoord, float]:
        """Current Aladin center and FOV for viewport-limited overlays."""
        target = self._aladin.target
        if isinstance(target, SkyCoord):
            coord = target
        else:
            ra, dec = target
            coord = SkyCoord(ra=ra, dec=dec, frame="icrs")
        fov = float(self._aladin.fov.to(u.deg).value)
        return coord, fov

    def _on_aladin_view_trait(self, _change) -> None:
        if not self._ready:
            return
        self._schedule_overlay_refresh()

    def _schedule_overlay_refresh(self) -> None:
        timer = self._overlay_refresh_timer
        if timer is not None:
            timer.cancel()
        self._overlay_refresh_timer = threading.Timer(0.35, self._refresh_overlay_from_aladin)
        self._overlay_refresh_timer.daemon = True
        self._overlay_refresh_timer.start()

    def _refresh_overlay_from_aladin(self) -> None:
        if not self._ready or not self.show_overlay:
            return
        try:
            self._refresh_overlay()
        except Exception as exc:
            self._overlay_status.object = f"**Overlay refresh failed:** `{exc}`"

    def _set_hips_survey(self, survey_name: str) -> None:
        url = hips_survey_url(survey_name)
        if url == self._hips_current_survey:
            return
        self._aladin.survey = url
        self._hips_current_survey = url

    def _refresh_overlay(
        self,
        coord: SkyCoord | None = None,
        *,
        fov_deg: float | None = None,
    ) -> None:
        view_coord, view_fov = self._overlay_view()
        if coord is None:
            coord = view_coord
        if fov_deg is None:
            fov_deg = view_fov

        catalog_key = str(self.catalog)
        catalog_name = catalog_overlay_name(catalog_key)
        fov = float(fov_deg)

        if not self.show_overlay:
            overlay_catalog_by_band(
                self._aladin,
                pd.DataFrame(),
                catalog_name,
                coord,
                fov,
                replace=True,
            )
            self._overlay_status.object = "_Catalog overlay off._"
            return

        try:
            df = load_catalog(catalog_key, self._catalog_cache)
        except (FileNotFoundError, KeyError) as exc:
            self._overlay_status.object = f"**Overlay failed:** `{exc}`"
            return

        result = overlay_catalog_by_band(
            self._aladin,
            df,
            catalog_name,
            coord,
            fov,
            max_rows=int(self.max_sources),
        )
        cap_note = f" (capped at {self.max_sources})" if result.truncated else ""
        cat_label = CATALOG_LABELS.get(catalog_key, catalog_key)
        self._overlay_status.object = (
            f"**Overlay:** {result.drawn} drawn, {result.in_fov} in FOV{cap_note} "
            f"— {cat_label} on `{self._hips_survey_w.value}`."
        )

    def _update_sky_view(self, coord: SkyCoord) -> None:
        self._set_hips_survey(str(self._hips_survey_w.value))
        self._aladin.target = coord
        self._aladin.fov = float(self._hips_fov_w.value)
        self._hips_status.object = (
            f"Centered on RA={coord.ra.deg:.6f}, Dec={coord.dec.deg:.6f} "
            f"(FOV={self._hips_fov_w.value:.2f}°)."
        )
        self._refresh_overlay(coord)

    def _on_catalog_change(self, _event=None) -> None:
        if not self._ready:
            return
        try:
            self._refresh_overlay()
        except Exception as exc:
            self._overlay_status.object = f"**Overlay refresh failed:** `{exc}`"

    def _on_hips_survey_change(self, _event=None) -> None:
        self._set_hips_survey(str(self._hips_survey_w.value))
        try:
            self._refresh_overlay()
        except Exception:
            pass

    def _on_hips_fov_change(self, _event=None) -> None:
        self._aladin.fov = float(self._hips_fov_w.value)
        try:
            self._refresh_overlay()
        except Exception:
            pass

    def _on_overlay_change(self, _event=None) -> None:
        if not self._ready:
            return
        try:
            self._refresh_overlay()
        except Exception as exc:
            self._overlay_status.object = f"**Overlay refresh failed:** `{exc}`"

    def _on_max_sources_change(self, _event=None) -> None:
        if not self._ready:
            return
        try:
            self._refresh_overlay()
        except Exception as exc:
            self._overlay_status.object = f"**Overlay refresh failed:** `{exc}`"

    def _on_center_click(self, _event=None) -> None:
        try:
            coord = parse_coordinate(self.coordinate)
        except Exception as exc:
            self._hips_status.object = f"**Center failed:** `{exc}`"
            return
        self._update_sky_view(coord)


hips_viewer = ReliabilityHiPSViewer()
hips_viewer


ReliabilityHiPSViewer(catalog='metacatalog', coordinate='83.633 -5.391', max_sources=1000, name='ReliabilityHiPSViewer00178', show_overlay=True)